In [1]:
import sys
import pandas as pd
import numpy as np
import torch
from datetime import datetime

from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments    
)

import optuna
from optuna.samplers import TPESampler

import matplotlib.pyplot as plt

sys.path.append('../src')

from dataset import RepositorioDados
from models.har import HarModel
from models.transformer import compute_metrics, evaluate_and_visualize

# Detecta o dispositivo e a precisão usada nas operações
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
print(f"Device: {device} | Precision: {dtype}")

# Seed para resultados reproduzíveis
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

c:\Users\Andre\OneDrive\Documentos\Github\iniciacao-cientifica\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu | Precision: torch.float32


In [2]:
# ==================== CONFIGURAÇÕES PRÉ-TREINAMENTO ====================
TIMESTAMP_COLUMN = 'timestamp'  # Coluna com timestamps em ms

# Tamanhos das janelas: histórico 7 dias, previsão 1 dia (dados diários)
CONTEXT_LENGTH = 512         # Janela histórica: x dias passados
FORECAST_HORIZON = 1        # Prever: x dias à frente

TRAIN_FRAC, VALID_FRAC = 0.7, 0.1  # Frações treino/validação/teste

# Hyperparâmetros do modelo
PATCH_LENGTH = 1            # Tamanho do patch (1=sem patchificação, mantém cada dia)
BATCH_SIZE = 32             # Samples por batch (reduzir se GPU memory limitada)
NUM_WORKERS = 0             # Workers para data loading (0 em Windows)
EPOCHS = 50                 # Reduzido: 50→30 (volatilidade tem ciclos curtos)
LEARNING_RATE = 1e-4        # Taxa de aprendizado

In [3]:
repo = RepositorioDados()

In [4]:
FEATURES = ["Vol_lag_1", "Vol_lag_2", "Vol_lag_3"]
TARGET_COLUMN = ['Vol']
ID_COLUMNS = []

In [22]:
tsp, train_ds, valid_ds, test_ds = repo.executar(
    timestamp_col=TIMESTAMP_COLUMN,
    train_frac=TRAIN_FRAC,
    valid_frac=VALID_FRAC,
    context_length=CONTEXT_LENGTH,
    features=FEATURES,
    target=TARGET_COLUMN,
    id_cols=ID_COLUMNS,
    forecast_horizon=FORECAST_HORIZON,
    use_mean_features=False,
    lags=3
)

Carregando dados de C:\Users\Andre\OneDrive\Documentos\Github\iniciacao-cientifica\data\raw\BTCUSDT_5m.txt
Calculando features de volatilidade
Treino: 1277 amostras | Val: 255 | Teste: 512


In [23]:
train_ds.datasets[0].data_df

,timestamp,Vol,Vol_lag_1,Vol_lag_2,Vol_lag_3,group
0,2017-12-03,1.479309,0.184190,0.703006,2.616550,0
1,2017-12-04,0.425121,1.479094,0.183629,0.700016,0
2,2017-12-05,-0.051191,0.424903,1.478379,0.181613,0
3,2017-12-06,0.850006,-0.051410,0.424314,1.473932,0
4,2017-12-07,3.131289,0.849789,-0.051943,0.421846,0
...,...,...,...,...,...,...
1784,2022-10-22,-0.413028,-0.344218,-0.361243,-0.380713,0
1785,2022-10-23,-0.382589,-0.413248,-0.344716,-0.362236,0
1786,2022-10-24,-0.370757,-0.382809,-0.413737,-0.345740,0
1787,2022-10-25,-0.260660,-0.370976,-0.383302,-0.414631,0


In [ ]:
# 1. Features
FEATURE_COMBINATIONS = {
    1: ["Vol_lag_1", "Vol_week_mean", "Vol_month_mean"],
    3: ["Vol_lag_1", "Vol_lag_2", "Vol_lag_3"],
}

# 2. Janelas temporais
CONTEXT_LENGTHS = [256, 512]
FORECAST_HORIZONS = [1]  # Manter fixo por enquanto

# 3. Hiperparâmetros do modelo
MODEL_PARAMS = {
    'd_model': [64, 128],
    'num_attention_heads': [8, 16],
    'num_hidden_layers': [2, 3],
    'ffn_dim': [256, 512],
    'dropout': [0.1, 0.2],
    'patch_length': [1, 16],
}

# 4. Hiperparâmetros de treinamento
TRAINING_PARAMS = {
    'learning_rate': [1e-4, 5e-4],
    'batch_size': [32, 64],
}

# Configurações fixas
FIXED_PARAMS = {
    'epochs': 30,  # Reduzido para grid search
    'early_stopping_patience': 5,
    'num_workers': 0,
    'train_frac': 0.7,
    'valid_frac': 0.1,
}

print("Espaços de busca definidos")
print(f"Total de combinações de features: {len(FEATURE_COMBINATIONS)}")
print(f"Total de combinações de context_length: {len(CONTEXT_LENGTHS)}")
print(f"Total de combinações de modelo: {np.prod([len(v) for v in MODEL_PARAMS.values()])}")
print(f"Total de combinações de treinamento: {np.prod([len(v) for v in TRAINING_PARAMS.values()])}")
total = len(FEATURE_COMBINATIONS) * len(CONTEXT_LENGTHS) * np.prod([len(v) for v in MODEL_PARAMS.values()]) * np.prod([len(v) for v in TRAINING_PARAMS.values()])
print(f"\n🔍 Total de experimentos: {int(total)}")

Espaços de busca definidos
Total de combinações de features: 4
Total de combinações de context_length: 3
Total de combinações de modelo: 64
Total de combinações de treinamento: 4

🔍 Total de experimentos: 3072


In [ ]:
def run_experiment(
    features,
    context_length,
    forecast_horizon,
    d_model,
    num_attention_heads,
    num_hidden_layers,
    ffn_dim,
    dropout,
    patch_length,
    learning_rate,
    batch_size,
    experiment_id,
    use_mean_features,
    lags
):
    """
    Executa um experimento completo com os parâmetros fornecidos.
    Retorna um dicionário com os resultados.
    
    use_fast_config: Se True, usa configurações otimizadas para busca rápida
    """
    print(f"\n{'='*80}")
    print(f"🧪 EXPERIMENTO {experiment_id}")
    print(f"{'='*80}")
    print(f"Features: {features}")
    print(f"Context Length: {context_length}")
    print(f"d_model: {d_model}, heads: {num_attention_heads}, layers: {num_hidden_layers}")
    print(f"LR: {learning_rate}, Batch: {batch_size}, Patch: {patch_length}")
    print(f"{'='*80}\n")
    
    try:
        # Escolher configuração (rápida ou completa)
        config_params = FIXED_PARAMS
        
        # 1. Preparar dados (patch já aplicado no repo.executar)
        tsp, train_ds, valid_ds, test_ds = repo.executar(
            timestamp_col=TIMESTAMP_COLUMN,
            train_frac=config_params['train_frac'],
            valid_frac=config_params['valid_frac'],
            context_length=context_length,
            features=features,
            target=TARGET_COLUMN,
            id_cols=ID_COLUMNS,
            forecast_horizon=forecast_horizon,
            use_mean_features=use_mean_features,
            lags=lags
        )
        
        # 2. Configurar modelo
        config = PatchTSTConfig(
            do_mask_input=False,
            context_length=context_length,
            patch_length=patch_length,
            num_input_channels=len(TARGET_COLUMN),
            patch_stride=patch_length,
            prediction_length=forecast_horizon,
            d_model=d_model,
            num_attention_heads=num_attention_heads,
            num_hidden_layers=num_hidden_layers,
            ffn_dim=ffn_dim,
            dropout=dropout,
            head_dropout=dropout,
            pooling_type=None,
            channel_attention=True,
            scaling='std',
            loss='mse',
            pre_norm=True,
            norm_type='batchnorm',
        )
        
        model = PatchTSTForPrediction(config=config).to(device).to(dtype)
        
        # 3. Configurar treinamento
        train_args = TrainingArguments(
            output_dir="./grid_search_temp",  # Pasta única para todos os experimentos
            overwrite_output_dir=True,
            learning_rate=learning_rate,
            num_train_epochs=config_params['epochs'],
            do_eval=True,
            eval_strategy="epoch",
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            dataloader_num_workers=config_params['num_workers'],
            save_strategy="no",  # Não salvar checkpoints durante busca
            logging_strategy="epoch",
            logging_dir=None,  # Sem logs individuais
            load_best_model_at_end=False,  # Não precisa carregar melhor modelo na busca
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            label_names=["future_values"],
            report_to="none",  # Desabilitar wandb/tensorboard
        )
        
        early_stopping = EarlyStoppingCallback(
            early_stopping_patience=config_params['early_stopping_patience'],
            early_stopping_threshold=0.001
        )
        
        trainer = Trainer(
            model=model,
            args=train_args,
            train_dataset=train_ds,
            eval_dataset=valid_ds,
            compute_metrics=compute_metrics,
            callbacks=[early_stopping]
        )
        
        # 4. Treinar
        train_result = trainer.train()
        
        # 5. Avaliar no conjunto de validação
        eval_result = trainer.evaluate()
        
        # 6. Coletar métricas
        result = {
            'experiment_id': experiment_id,
            'features': str(features),
            'num_features': len(features),
            'context_length': context_length,
            'forecast_horizon': forecast_horizon,
            'd_model': d_model,
            'num_attention_heads': num_attention_heads,
            'num_hidden_layers': num_hidden_layers,
            'ffn_dim': ffn_dim,
            'dropout': dropout,
            'patch_length': patch_length,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'train_loss': train_result.training_loss,
            'eval_loss': eval_result['eval_loss'],
            'eval_MSE': eval_result.get('eval_MSE', None),
            'eval_MAE': eval_result.get('eval_MAE', None),
            'eval_RMSE': eval_result.get('eval_RMSE', None),
            'eval_MAPE': eval_result.get('eval_MAPE', None),
            'epochs_trained': train_result.global_step // len(train_ds) * batch_size,
            'status': 'success'
        }
        
        print(f"✅ Experimento {experiment_id} concluído com sucesso!")
        print(f"   Val Loss: {eval_result['eval_loss']:.6f} | RMSE: {result['eval_RMSE']:.6f}")
        
        # Limpar memória
        del model, trainer, train_ds, valid_ds, test_ds, tsp
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        return result
        
    except Exception as e:
        print(f"❌ Erro no experimento {experiment_id}: {str(e)}")
        return {
            'experiment_id': experiment_id,
            'features': str(features),
            'context_length': context_length,
            'd_model': d_model,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'status': 'failed',
            'error': str(e)
        }

print("✓ Função run_experiment definida (com modo rápido)")

In [ ]:
def optuna_objective(trial):
    """
    Função objetivo para Optuna.
    Retorna a métrica a ser MINIMIZADA (eval_loss ou RMSE).
    """
    # Sugere hiperparâmetros a partir dos espaços definidos na célula 6
    features_idx = trial.suggest_categorical('features_idx', list(FEATURE_COMBINATIONS.keys()))
    features = FEATURE_COMBINATIONS[features_idx]

    context_length = trial.suggest_categorical('context_length', CONTEXT_LENGTHS)
    forecast_horizon = trial.suggest_categorical('forecast_horizon', FORECAST_HORIZONS)

    d_model = trial.suggest_categorical('d_model', MODEL_PARAMS['d_model'])
    num_attention_heads = trial.suggest_categorical('num_attention_heads', MODEL_PARAMS['num_attention_heads'])
    num_hidden_layers = trial.suggest_categorical('num_hidden_layers', MODEL_PARAMS['num_hidden_layers'])
    ffn_dim = trial.suggest_categorical('ffn_dim', MODEL_PARAMS['ffn_dim'])
    dropout = trial.suggest_categorical('dropout', MODEL_PARAMS['dropout'])
    patch_length = trial.suggest_categorical('patch_length', MODEL_PARAMS['patch_length'])

    learning_rate = trial.suggest_categorical('learning_rate', TRAINING_PARAMS['learning_rate'])
    batch_size = trial.suggest_categorical('batch_size', TRAINING_PARAMS['batch_size'])

    # Executar experimento
    result = run_experiment(
        features=features,
        context_length=context_length,
        forecast_horizon=forecast_horizon,
        d_model=d_model,
        num_attention_heads=num_attention_heads,
        num_hidden_layers=num_hidden_layers,
        ffn_dim=ffn_dim,
        dropout=dropout,
        patch_length=patch_length,
        learning_rate=learning_rate,
        batch_size=batch_size,
        experiment_id=trial.number,
        use_fast_config=True
    )

    # Salvar resultado completo como atributo do trial
    trial.set_user_attr('full_result', result)

    # Se falhou, retornar valor alto
    if result['status'] == 'failed':
        return float('inf')

    # Retornar métrica a minimizar
    return result['eval_RMSE']  # ou 'eval_loss'

In [ ]:
study = optuna.create_study(
    direction='minimize',  # Minimizar RMSE
    sampler=TPESampler(seed=RANDOM_STATE),
    study_name='patchtst_optimization'
)

In [ ]:
start_time = datetime.now()

# Executar otimização
study.optimize(
    optuna_objective,
    n_trials=N_RANDOM_TRIALS,
    show_progress_bar=True,
    callbacks=[
        lambda study, trial: study.trials_dataframe().to_csv(
            'optuna_results_partial.csv', index=False
        ) if trial.number % 5 == 0 else None
    ]
)

end_time = datetime.now()
duration = end_time - start_time

print(f"\n{'='*80}")
print(f"✅ Busca Optuna concluída!")
print(f"⏱️ Tempo total: {duration}")
print(f"🏆 Melhor RMSE: {study.best_value:.6f}")
print(f"{'='*80}\n")

# Extrair resultados
results = [trial.user_attrs.get('full_result') for trial in study.trials 
            if 'full_result' in trial.user_attrs]

# Salvar resultados
df_results = pd.DataFrame(results)
df_results.to_csv('optuna_results.csv', index=False)

# Mostrar melhores parâmetros
print("🏆 MELHORES HIPERPARÂMETROS (Optuna):")
print("="*80)
for key, value in study.best_params.items():
    print(f"{key:25s}: {value}")
print("="*80)

# Visualização Optuna
try:
    from optuna.visualization import plot_optimization_history, plot_param_importances
    
    # Histórico de otimização
    fig1 = plot_optimization_history(study)
    fig1.write_image('optuna_history.png')
    fig1.show()
    
    # Importância dos parâmetros
    fig2 = plot_param_importances(study)
    fig2.write_image('optuna_importance.png')
    fig2.show()
    
    print("📊 Gráficos salvos: optuna_history.png, optuna_importance.png")
except Exception as e:
    print(f"⚠️ Não foi possível gerar visualizações Optuna: {e}")
    print("   Instale: pip install plotly kaleido")